<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/sistema_graphrag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema GraphRAG — Leyes de inversión (Neo4j + OpenAI)

- **Setup** (compartido): dependencias, credenciales, config, driver, y objetos compartidos (embedder, LLM, prompt).
- **Parte A — Backfill** (correr *una sola vez*): embeddings + índices.
- **Parte B — Retriever GraphRAG** (uso normal): recuperación con expansión al grafo + pipeline.
- **Parte C — Baseline y comparación**: mismo sistema con el grafo apagado.
- **Parte D — Conversación con memoria** (LangGraph, Etapa 1): chat multi-turno que reusa el pipeline.

**Antes de correr:** en **Colab → Secrets** cargá (con acceso a este notebook): `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.

**Orden:** Setup → (Parte A si hace falta) → Parte B → Parte C / Parte D.

## Setup (compartido)

Todas las dependencias se instalan en una sola celda (evita choques de versión). El `embedder`, el `llm` y el `prompt_template` se definen acá para que los usen las Partes B, C y D.

> ⚠ **Si venís de una corrida con error de imports** (por ejemplo tras haber instalado algo a mitad de sesión), hacé **Runtime → Reiniciar sesión** y volvé a correr desde esta celda. Con este notebook, al instalar todo junto arriba, no debería hacer falta.

In [1]:
# Dependencias (todo en una sola instalacion coherente, para evitar choques de version).
# neo4j-graphrag trae el driver de neo4j y openai; langgraph trae langchain-core.
!pip install -q "neo4j_graphrag[openai]" langgraph tiktoken tenacity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 12.2 MB/s eta 0:00:00


In [2]:
# Credenciales desde Colab Secrets (o por teclado fuera de Colab)
import os
_KEYS = ["OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE"]
try:
    from google.colab import userdata
    for k in _KEYS:
        os.environ[k] = userdata.get(k)
    print("Credenciales cargadas desde Colab Secrets.")
except Exception:
    import getpass
    for k in _KEYS:
        if not os.environ.get(k):
            os.environ[k] = getpass.getpass(f"{k}: ")
    print("Credenciales cargadas por teclado.")

Credenciales cargadas desde Colab Secrets.


### Configuración

`EMBEDDING_MODEL` es el mismo para backfill y retriever. `TEST_LIMIT=20` deja la Parte A en modo prueba; poné `None` para el corpus completo.

In [3]:
import os, time
from openai import OpenAI
from neo4j import GraphDatabase
import tiktoken
from tenacity import retry, wait_random_exponential, stop_after_attempt

# (opcional) silenciar el warning de deprecacion de db.index.vector.queryNodes:
import logging; logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

# --- Conexion ---
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USER     = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]     # tu base: ece63d51

# --- Modelos ---
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
EMBEDDING_MODEL      = "text-embedding-3-small"   # backfill Y retriever: deben coincidir
EMBEDDING_DIM        = 1536
EMBEDDING_ENCODING   = "cl100k_base"
MAX_TOKENS_PER_INPUT = 8000
LLM_MODEL            = "gpt-4o"                    # comparable con ALIBot (Belen)

# --- Esquema del grafo ---
LABEL_ARTICULO    = "Articulo"
LABEL_RECUPERABLE = "Recuperable"
REL_TIENE_ART     = "TIENE_ARTICULO"
NORM_LABELS       = ["Ley", "Decreto", "Resolucion"]
PROP_TEXTO        = "texto"
PROP_NUM_ART      = "numero"
PROP_PARTE        = "parte"
PROP_LEY_NUMERO   = "ley_numero"
PROP_NUM_NORMA    = "numero"
PROP_NOMBRE_COMPLETO = "nombre_completo"
PROP_EMBEDDING    = "embedding"
PROP_ES_PRUEBA    = "es_prueba"

# --- Indices ---
VECTOR_INDEX_NAME   = "recuperable_embedding"
FULLTEXT_ART_NAME   = "articulo_texto_ft"
FULLTEXT_NORMA_NAME = "normas_nombre_ft"

# --- Tunables ---
EMBED_BATCH_SIZE = 50
WRITE_BATCH_SIZE = 500
TEST_LIMIT       = None     # backfill: 20 prueba; None = corpus completo
TOP_K            = 5      # retriever

# --- Clientes y driver COMPARTIDOS ---
client = OpenAI(api_key=OPENAI_API_KEY)          # usado por el backfill
enc    = tiktoken.get_encoding(EMBEDDING_ENCODING)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Setup listo (config + clientes + driver).")

Setup listo (config + clientes + driver).


### Objetos compartidos (embedder, LLM, prompt)

In [4]:
# Objetos COMPARTIDOS: embedder, LLM y prompt.
# Los usan el retriever GraphRAG, el baseline y (mas adelante) LangGraph.
# Definirlos aca, en el Setup, evita que una celda dependa de que otra se haya corrido antes.
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.generation import RagTemplate

embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL)   # MISMO modelo que el backfill
llm = OpenAILLM(model_name=LLM_MODEL, model_params={"temperature": 0})

TEMPLATE = """Sos un asistente legal sobre derecho de inversiones de Paraguay.
Respondé usando SOLO el contexto. Citá la norma y el artículo en cada afirmación.
Si una norma figura como DEROGADA o con estado distinto de vigente, aclaralo
explícitamente y NO la presentes como vigente. Si el contexto no alcanza, decilo.

# Contexto:
{context}
{examples}
# Pregunta:
{query_text}

# Respuesta:"""
prompt_template = RagTemplate(template=TEMPLATE)
print("Embedder, LLM y prompt listos (compartidos).")

Embedder, LLM y prompt listos (compartidos).


## Parte A1 — Backfill (correr una sola vez)

Crea los índices y genera los embeddings. **Idempotente**: saltea lo ya embebido. Si ya lo corriste antes, podés saltear esta parte.

In [ ]:
def crear_indices(driver):
    driver.execute_query(
        f"""
        CREATE VECTOR INDEX {VECTOR_INDEX_NAME} IF NOT EXISTS
        FOR (n:{LABEL_RECUPERABLE}) ON (n.{PROP_EMBEDDING})
        OPTIONS {{ indexConfig: {{
            `vector.dimensions`: {EMBEDDING_DIM},
            `vector.similarity_function`: 'cosine'
        }} }}
        """,
        database_=NEO4J_DATABASE,
    )
    driver.execute_query(
        f"CREATE FULLTEXT INDEX {FULLTEXT_ART_NAME} IF NOT EXISTS "
        f"FOR (a:{LABEL_ARTICULO}) ON EACH [a.{PROP_TEXTO}]",
        database_=NEO4J_DATABASE,
    )
    normas = "|".join(NORM_LABELS)
    driver.execute_query(
        f"CREATE FULLTEXT INDEX {FULLTEXT_NORMA_NAME} IF NOT EXISTS "
        f"FOR (n:{normas}) ON EACH [n.{PROP_NOMBRE_COMPLETO}]",
        database_=NEO4J_DATABASE,
    )
    print("Indices creados/verificados.")


def leer_pendientes(driver):
    norm_pred = " OR ".join(f"n:{l}" for l in NORM_LABELS)
    limit_clause = f"LIMIT {TEST_LIMIT}" if TEST_LIMIT else ""
    query = f"""
        MATCH (a:{LABEL_ARTICULO})
        WHERE a.{PROP_TEXTO} IS NOT NULL AND trim(a.{PROP_TEXTO}) <> ''
              AND a.{PROP_EMBEDDING} IS NULL
              AND coalesce(a.{PROP_ES_PRUEBA}, false) = false
        OPTIONAL MATCH (n)-[:{REL_TIENE_ART}]->(a)
            WHERE {norm_pred}
        WITH a, collect(n)[0] AS norm
        RETURN elementId(a) AS eid,
               a.{PROP_NUM_ART}    AS art_num,
               a.{PROP_PARTE}      AS parte,
               a.{PROP_TEXTO}      AS texto,
               a.{PROP_LEY_NUMERO} AS ley_numero,
               CASE WHEN norm IS NULL THEN null
                    ELSE head([l IN labels(norm) WHERE l IN {NORM_LABELS}]) END AS norm_tipo,
               norm.{PROP_NUM_NORMA} AS norm_num
        {limit_clause}
    """
    records, _, _ = driver.execute_query(query, database_=NEO4J_DATABASE)
    return [r.data() for r in records]


def texto_con_encabezado(row):
    partes = []
    tipo = row.get("norm_tipo")
    num = row.get("norm_num") or row.get("ley_numero")
    if tipo and num:
        partes.append(f'{tipo} {num}')
    elif num:
        partes.append(f'{num}')
    if row.get("art_num") is not None:
        partes.append(f'Artículo {row["art_num"]}')
    if row.get("parte") and row["parte"] not in (None, "", "cuerpo"):
        partes.append(f'({row["parte"]})')
    prefijo = ", ".join(partes)
    return f'{prefijo}: {row["texto"]}' if prefijo else row["texto"]


def recortar_a_limite(texto):
    toks = enc.encode(texto)
    if len(toks) <= MAX_TOKENS_PER_INPUT:
        return texto, len(toks), False
    return enc.decode(toks[:MAX_TOKENS_PER_INPUT]), MAX_TOKENS_PER_INPUT, True


@retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(6))
def embeber_lote(textos):
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=textos)
    return [d.embedding for d in resp.data]


def escribir_vectores(driver, filas):
    query = f"""
        UNWIND $rows AS row
        MATCH (a) WHERE elementId(a) = row.eid
        CALL db.create.setNodeVectorProperty(a, $prop, row.embedding)
        SET a:{LABEL_RECUPERABLE}
    """
    for i in range(0, len(filas), WRITE_BATCH_SIZE):
        lote = filas[i:i + WRITE_BATCH_SIZE]
        driver.execute_query(query, rows=lote, prop=PROP_EMBEDDING, database_=NEO4J_DATABASE)


def verificar(driver):
    q = f"""
        MATCH (a:{LABEL_ARTICULO})
        RETURN count(a) AS total,
               count(a.{PROP_EMBEDDING}) AS con_embedding,
               sum(CASE WHEN a.{PROP_TEXTO} IS NOT NULL
                        AND a.{PROP_EMBEDDING} IS NULL THEN 1 ELSE 0 END) AS pendientes
    """
    records, _, _ = driver.execute_query(q, database_=NEO4J_DATABASE)
    print("Verificacion:", records[0].data())

#Parte A2 - Cargar FICHAS (documentos operativos MIC/REDIEX)

In [ ]:
# =============================================================================
# PARTE A2 — Cargar FICHAS (documentos operativos MIC/REDIEX)
# =============================================================================
# Las fichas aportan lo que el articulado NO tiene: REQUISITOS, PROCESO y
# beneficios explicados. Se cargan como nodos :Ficha con etiqueta :Recuperable
# (entran al MISMO indice vectorial, sin recrear nada) y se enlazan con
# (Ficha)-[:DESCRIBE]->(norma).
#
# CLAVE: el retrieval_query expande por DESCRIBE y trae la VIGENCIA de cada
# norma descrita -> el grafo mantiene HONESTAS a las fichas. Varias fichas
# oficiales del MIC citan normas derogadas (ej. la 60/90); el sistema lo marca.
#
# Requisitos: haber corrido el Setup (driver, client, enc, EMBEDDING_MODEL).
# Es IDEMPOTENTE: se puede re-correr sin duplicar ni re-embeber.
# =============================================================================

FICHAS = [
    # -------------------------------------------------------------------------
    # HISTÓRICA: describe la Ley 60/90, DEROGADA por la 7548/25.
    # Se carga porque la 7548/25 SÍ está en el grafo (puede corregir), y porque
    # su Art. 35 mantiene las reglas viejas para proyectos ya aprobados.
    # -------------------------------------------------------------------------
    {
        "id": "ficha_incentivos_fiscales_60_90",
        "titulo": "Régimen de Incentivos Fiscales para la Inversión de Capital (Ley 60/90)",
        "tipo": "ficha_mic",
        "estado": "historica",
        "fuente": "MIC",
        "nota": ("Describe el régimen de la Ley 60/90, DEROGADA por la Ley 7548/25. "
                 "Se conserva por valor transicional: el Art. 35 de la Ley 7548/25 mantiene "
                 "las disposiciones anteriores para los proyectos ya acogidos a la 60/90."),
        "describe": ["60/90", "22031"],
        "texto": """OBJETO: Promover e incrementar las inversiones de capital de origen nacional y/o extranjero que tengan por objeto: acrecentar la producción de bienes y servicios; crear fuentes de trabajo permanente; fomentar las exportaciones y sustituir importaciones; incorporar tecnologías que permitan aumentar la eficiencia productiva y mayor utilización de materias primas, mano de obra y recursos energéticos nacionales; y la reinversión de utilidades en bienes de capital. Beneficia a todos los sectores: Industrial, Agropecuario, Minas y Canteras, y Servicios.

BENEFICIOS QUE OFRECE:
- Arancel 0% para importación de bienes de capital (maquinarias y equipos que no son fabricados en Paraguay).
- Impuesto al Valor Agregado (IVA) 0% sobre bienes de capital (que no son fabricados en Paraguay cuando se trata de importaciones, y que son fabricados en Paraguay cuando se trata de compras locales).
- Exoneración del impuesto aplicado a las remesas y pagos en concepto de intereses para inversiones mayores a 5 millones de US$.
- Exoneración del impuesto sobre las remesas de dividendos y utilidades para inversiones mayores a 5 millones de US$ por 10 años, siempre que no provenga de un territorio de baja o nula tributación o no sea crédito fiscal en el país inversor.

REQUISITOS: Presentar por Nota solicitud ante el Ministerio de Industria y Comercio (MIC), acompañada de:
- Proyecto de inversión e Informe de Inversión en caso de reinversión.
- Constitución de sociedad.
- Acta de Asamblea.
- Antecedentes judiciales de directores y cédula de identidad del representante legal.
- Título de propiedad coincidente con la Licencia Ambiental.
- Autorización de organismos competentes (permisos habilitantes conforme al sector).
- Certificado de funcionamiento de bienes de capital (cuando la antigüedad del bien supere los 5 años de fabricación).
- Constancia de RUC (Subsecretaría de Estado de Tributación - SET).
- Certificado de cumplimiento tributario y del seguro social (SET e IPS respectivamente).
- Constancia en el Registro de Personas Jurídicas y de Beneficiarios Finales (Abogacía del Tesoro).
- Despacho de importación (en caso de importación provisoria).
- Estados financieros (3 últimos ejercicios cerrados: Balance General, Estado de Resultados, Estado de flujo de efectivo, cambios del patrimonio neto y notas).
- Facturas pro forma de bienes de capital a importar y/o compra local.
- Inscripción en el Banco Central del Paraguay (empresas con capital extranjero).
- Licencia ambiental (Ministerio del Ambiente).
- Autorización de la SET (formato DDI).
- Referencia bancaria.
- RIEL activo para empresas que están operando; las que están por iniciar operaciones deben inscribirse en un plazo no mayor a 6 meses luego de la importación de bienes de capital.
- Contrato o constancia de la entidad financiera que proveerá el crédito (si solicita el inciso "f" del Art. 5º de la Ley Nº 60/90).

PROCESO: La Dirección del Viceministerio de Industria realiza el chequeo documental y eleva el expediente al Consejo de Inversiones (órgano mixto público-privado), que dictamina favorablemente o solicita información complementaria. En caso de dictamen favorable, los beneficios son otorgados por Resolución biministerial suscrita por el Ministro de Industria y Comercio (MIC) y el Ministro de Hacienda (MH). El organismo de aplicación es el MIC, y el MH está a cargo de los aspectos tributarios.

LEGISLACIÓN RESPALDATORIA: Ley Nº 60/90; Decreto reglamentario Nº 22.031/2003, modificado por el Decreto Nº 6427/2005 y el Decreto Nº 11.462/2013.""",
    },

    # -------------------------------------------------------------------------
    # VIGENTE. Ojo: la ficha cita incentivos de la Ley 60/90 (DEROGADA) ->
    # el grafo lo marcará vía DESCRIBE.
    # -------------------------------------------------------------------------
    {
        "id": "ficha_zonas_francas_523_95",
        "titulo": "Régimen de Zonas Francas (Ley 523/95)",
        "tipo": "ficha_mic",
        "estado": "vigente",
        "fuente": "MIC",
        "nota": ("La ficha menciona la concesión de incentivos fiscales de la Ley 60/90, "
                 "que está DEROGADA por la Ley 7548/25."),
        "describe": ["523/95", "60/90"],
        "texto": """OBJETO: Promover la atracción de inversión productiva, diversificar las exportaciones, la generación de empleo y la transferencia de conocimiento y especialización de mano de obra paraguaya.

ALCANCE: Las Zonas Francas son espacios del territorio nacional, sujetas al control fiscal, aduanero y administrativo, en las cuales se pueden desarrollar actividades comerciales, industriales y de servicios. Es CONCESIONARIO la persona jurídica que, mediante contrato celebrado con el Poder Ejecutivo, adquiere el derecho de habilitar, administrar y explotar una Zona Franca, otorgado por 30 años de plazo, prorrogables. Es USUARIO la persona física o jurídica que desarrolla actividades comerciales, industriales y/o de servicios dentro de la Zona Franca.

BENEFICIOS QUE OFRECE:
- Importación de materia prima o mercaderías con una tasa de 0%.
- Impuesto al Valor Agregado (IVA) 0%.
- Servicios y comercios entre usuarios de Zonas Francas: 0% de impuestos.
- Exportación a terceros países: 0,5% del valor factura.
- Emisión de certificados de origen para los productos fabricados en Zona Franca que cumplan con los requisitos de origen MERCOSUR (ROM).
- Administración de Aduanas (DNA) instalada en la Concesionaria para agilizar los trámites de tránsito, introducción, importación y exportación.
- Disponibilidad de infraestructura inmobiliaria para todas las actividades.
- Alta disponibilidad de energía eléctrica de calidad a costo competitivo.
- Bajo costo de operación para fabricar y vender a clientes de Paraguay o países vecinos.
- Sin pérdida de origen de los productos introducidos en la zona franca (Ley Nº 523/95, Art. 20 y Decreto Nº 7068/2006).
- No requiere contratar póliza de seguro para garantías aduaneras.
- Concesión de incentivos fiscales de la Ley Nº 60/90 a la inversión nacional y extranjera.

REQUISITOS PARA SER CONCESIONARIO: El postulante deberá presentar ante el Consejo Nacional de Zonas Francas (CNZF) un proyecto de inversión que demuestre fehacientemente su viabilidad económica, conteniendo (Decreto Nº 15554/96, Art. 23):
a. Determinación de la forma o modalidad jurídica de la empresa a través de la cual se realizará la explotación.
b. La localización del predio y la superficie en que se propone desarrollar el proyecto.
c. Causas y consecuencias de su emplazamiento.
d. La posibilidad de su expansión futura.
e. Los servicios que se propone suministrar y el monto de inversión en servicios, indicando las responsabilidades de ejecución.
f. Descripción de las inversiones en infraestructura (caminos, cercado, construcciones, etc.).
g. Fuentes de financiamiento.
h. Tiempo estimado de realización del proyecto y fecha de comienzo de las obras; si se desarrolla por etapas, la superficie, obras e infraestructura de cada etapa y su tiempo de realización.
i. Estudio de mercado con indicación de cantidad y calidad de posibles Usuarios.
j. Estimación del personal a utilizar, tanto en el funcionamiento de la Zona como por parte de las empresas a instalarse.
k. Previsiones para el tratamiento de efluentes, eliminación de residuos y medidas de protección del medio ambiente.
l. Estimación del precio a cobrar a los Usuarios por el alquiler y/o venta de predios y construcciones.
m. Requerimiento de obras de infraestructura de apoyo por parte del Gobierno o empresas estatales, departamentales o municipales (caminos de acceso, puertos, tendido eléctrico, telefónico, etc.).
Los predios donde se instale la Zona Franca deberán ser propiedad del Concesionario o existir una relación contractual entre este y el propietario, por un plazo mínimo igual al fijado para la concesión (Decreto Nº 15554/96, Art. 15).

REQUISITOS PARA SER USUARIO:
- Contrato celebrado con el Concesionario.
- Inscripción en los registros nacionales de constitución de sociedad, RUC y patentes.
- Certificado de no hallarse en quiebra y de no tener inhibición de bienes.

PROCESO PARA LA CONCESIÓN DE UNA ZONA FRANCA: El postulante presenta ante el CNZF un proyecto de inversión conforme al Decreto Nº 15554/96. El CNZF estudia el proyecto y, con dictamen fundado, lo eleva al Poder Ejecutivo. De ser aprobado, se suscribe el contrato entre el Poder Ejecutivo y el postulante a Concesionario. Una vez inscripto, queda habilitado para comenzar las obras, pudiendo ingresar libre de tributos los materiales, bienes y equipos necesarios.

PROCESO PARA SER USUARIO DE UNA ZONA FRANCA: Se presenta la solicitud ante el Concesionario de la Zona Franca conforme a los requisitos establecidos. Estos se presentan a la Dirección Ejecutiva del CNZF, adjuntando el contrato respectivo y las demás documentaciones. La Dirección Ejecutiva expide la Constancia de Usuario, una vez cumplidos los requisitos, en un plazo no mayor a 48 horas hábiles.

LEGISLACIÓN RESPALDATORIA: Ley Nº 523/1995; Decreto reglamentario Nº 15554/1996; Decreto Nº 19461/2002; Decreto Nº 21309/2003; Decreto Nº 952/2018; Decreto Nº 4611/2020; Resolución General Nº 80/2021 de la SET.""",
    },

    # -------------------------------------------------------------------------
    # Ley 1064/97: derogación DIFERIDA (vigente_hasta) -> el grafo lo marcará.
    # -------------------------------------------------------------------------
    {
        "id": "ficha_maquila_1064_97",
        "titulo": "Régimen de Maquila (Ley 1064/97)",
        "tipo": "ficha_mic",
        "estado": "vigente",
        "fuente": "MIC",
        "nota": ("Verificar la vigencia de la Ley 1064/97 en el grafo: tiene derogación "
                 "diferida (vigente_hasta). La ficha no lo menciona."),
        "describe": ["1064/97", "9585"],
        "texto": """OBJETO: Promover el establecimiento y regular las operaciones de empresas industriales maquiladoras que se dediquen total o parcialmente a realizar procesos industriales o de servicios, incorporando mano de obra y otros recursos nacionales destinados a la transformación, elaboración, reparación o ensamblaje de mercaderías de procedencia extranjera importadas temporalmente a dicho efecto, para su reexportación posterior, en ejecución de un contrato suscrito con una empresa domiciliada en el extranjero.

ALCANCE: Beneficia a cualquier persona física o jurídica, nacional o extranjera, legalmente constituida en Paraguay, que se encuentre habilitada para realizar actos de comercio y que guarde relación comercial con otra empresa en el exterior. También a personas físicas o jurídicas con capacidad ociosa. Las industrias maquiladoras podrán instalarse en cualquier lugar del territorio paraguayo, adecuándose a los requisitos locales según el caso.

BENEFICIOS QUE OFRECE:
- 1% de tributo único sobre el valor agregado en territorio nacional, o sobre el valor de la factura emitida por orden y cuenta de la matriz, el que resulte mayor (Tributo Único Maquila).
- Suspensión de aranceles e impuestos a la importación de materias primas, insumos y bienes de capital.
- Exoneración de las tasas aduaneras, portuarias y aeroportuarias.
- Exoneración de tributos que gravan la remesa de dinero relacionada al régimen de maquila.
- Recuperación del crédito fiscal (IVA) correspondiente a la adquisición de bienes y servicios aplicados en forma directa o indirecta a las operaciones de maquila.
- Exoneración de impuestos departamentales o municipales (maquiladoras puras).

REQUISITOS: El trámite de inscripción al régimen de maquila es electrónico y se realiza a través de la plataforma VUE, en el módulo exclusivo para el régimen de maquila, adjuntando:
- Escritura Pública de Constitución (para empresas).
- Constancia de Inscripción en el Registro de Beneficiarios Finales y Personas Jurídicas.
- Documento de identidad de las personas físicas que solicitan su inscripción, o de los representantes de las personas jurídicas.
- Constancia de RUC.
- RUC de la empresa o persona.
- Acta actualizada de designación de directorio.

El PROGRAMA DE MAQUILA deberá contener, entre otros datos:
- Datos del solicitante.
- Características del programa de maquila, el tipo de programa a implementar y la forma de operación.
- Datos relativos a la actividad a desarrollar o servicios a prestar.
- Datos sobre la mano de obra a generar, materias primas, insumos y maquinarias a utilizar en el proceso maquilador, mercados de proveedores y de destino de los productos maquilados, exportación, importación y valor agregado nacional.
El programa deberá cargarse en la plataforma VUE acompañado de los recaudos documentales correspondientes.

PROCESO: Los interesados solicitan los beneficios del régimen a través de la plataforma VUE, pidiendo usuario y contraseña para el Régimen de Maquila; los procesos de inscripción y aprobación del programa de maquila son electrónicos. El Programa de Maquila es tratado por el Consejo Nacional de Industrias Maquiladoras de Exportación (CNIME) y, en caso de dictamen favorable, la decisión se formaliza por Resolución Biministerial (MIC-MH). Tiempo aproximado de duración del trámite: 90 días.

LEGISLACIÓN RESPALDATORIA: Ley Nº 1064/97; Decreto reglamentario Nº 9585/2000.""",
    },

    # -------------------------------------------------------------------------
    # VIGENTE. Ojo: cita requisitos y beneficios de la Ley 60/90 (DEROGADA).
    # -------------------------------------------------------------------------
    {
        "id": "ficha_garantia_inversiones_5542_15",
        "titulo": "Régimen de Garantía de Inversiones (Ley 5542/2015)",
        "tipo": "ficha_mic",
        "estado": "vigente",
        "fuente": "MIC",
        "nota": ("La ficha remite a los beneficios y requisitos de la Ley 60/90, "
                 "que está DEROGADA por la Ley 7548/25."),
        "describe": ["5542/15", "60/90"],
        "texto": """OBJETO: La protección de la inversión de capital en la creación de industrias u otras actividades productivas, cuando ellas contribuyan a la generación de empleo y al desarrollo económico y social, principalmente a través de la incorporación de valor agregado a la materia prima paraguaya o importada.

ALCANCE: Beneficia a las personas físicas o jurídicas, nacionales o extranjeras, que inviertan capital para la creación de empresas o que adquieran empresas existentes y cumplan con los objetivos mencionados.

OBLIGACIONES DE LA EMPRESA:
- Incorporar la totalidad del capital en el plazo establecido en el contrato.
- Sujeción a la legislación nacional, y en particular a las disposiciones en materia ambiental y de salud pública.
- Fiel cumplimiento del contrato.
- Someter los estados financieros anuales a auditoría externa.
- Presentar una declaración sobre las inversiones efectuadas en un año y abonar un canon anual equivalente al 1% (uno por ciento) de dicha inversión.

BENEFICIOS QUE OFRECE:
- Invariabilidad de la tasa impositiva del impuesto a la renta que grava la actividad desarrollada: por un plazo de hasta 10 años para inversiones menores a 50 millones de US$; 15 años para inversiones entre 50 y 100 millones de US$; y 20 años para inversiones de 100 millones de US$ y más.
- Libre transferencia de remesas de capital (a los 2 años desde la puesta en marcha) y de utilidades líquidas sin límite de tiempo.
- Régimen arancelario correspondiente a la importación de maquinarias y equipos que no se produzcan en el país, conforme a los beneficios de la Ley Nº 60/90 (arancel 0% para bienes de capital y 0% de IVA sobre los mismos).
- Régimen especial para la exportación: podrán mantener un porcentaje de divisas en el exterior cuando sean necesarias para pagar obligaciones legalmente autorizadas o para cumplir con la remesa de utilidades.
- Invariabilidad tributaria para la compra de empresas existentes o cuando se transfiera parte de sus acciones.
- Beneficios adicionales para las industrias de alto contenido social y sus accionistas: exoneración de la tasa adicional del 5% del impuesto a la renta por la distribución de utilidades; y disminución de la tasa impositiva aplicada a la remisión de utilidades al exterior en un 1% por cada 100 empleos directos generados, hasta un máximo del 50% del valor total de la tasa aplicable.
- Seguridad jurídica: las inversiones no podrán ser objeto de ninguna modalidad de apropiación ni confiscación, y están protegidas por el principio de irretroactividad de la ley.

REQUISITOS: Presentar solicitud por Nota ante el Ministerio de Industria y Comercio, acompañada de todos los requisitos previstos para ser beneficiaria de la Ley Nº 60/90 (proyecto y cronograma de inversión) y además:
- Nota de autorización al Equipo Económico Nacional, al Consejo de Inversiones, al Ministerio de Industria y Comercio (MIC) y al Ministerio de Hacienda (MH) para requerir y obtener información y datos obrantes en instituciones públicas y privadas, nacionales y extranjeras.
- Certificado de cumplimiento tributario y Constancia de RUC (SET).
- Certificado de cumplimiento con el seguro social (IPS) o constancia de inscripción patronal.
- Últimos 3 Estados Financieros cerrados, o Balance de Apertura para empresas nuevas.
- Referencia bancaria.
- Inscripción de Inversión Extranjera Directa (Banco Central del Paraguay).
- Autorización de los organismos competentes (permisos habilitantes conforme al sector).
- Escritura de constitución.
- Acta de la última Asamblea.
- Título de propiedad del inmueble coincidente con la licencia ambiental.
- Contrato o constancia de la entidad financiera que proveerá el crédito (si solicita el inciso "f" del Art. 5º de la Ley Nº 60/90).
- Facturas, proformas o despacho de importación (si solicita el inciso "c" del Art. 5º de la Ley Nº 60/90).
- Constancia en el Registro de Personas Jurídicas y de Beneficiarios Finales (Abogacía del Tesoro).
- Antecedentes judiciales de directores y cédula de identidad del representante legal.
- Licencia ambiental por la actividad del proyecto.
- Autorización de la SET (formato DDI).

PROCESO: La Dirección del Viceministerio de Industria realiza el chequeo documental y eleva el expediente al Consejo de Inversiones (órgano mixto público-privado), que dictamina favorablemente o solicita información complementaria. En caso de dictamen favorable, se eleva al Equipo Económico Nacional; si este aprueba el proyecto, se instrumentaliza a través de una Resolución biministerial suscrita por el Ministro de Industria y Comercio (MIC) y el Ministro de Hacienda (MH). El contrato es suscrito entre la empresa beneficiaria y el Ministro de Industria y Comercio en representación del Estado paraguayo. El organismo de aplicación es el MIC, y el MH está a cargo de los aspectos tributarios.

MARCO LEGAL: Ley Nº 5542/2015; Decreto reglamentario Nº 6100/2016.""",
    },

    # -------------------------------------------------------------------------
    # La ficha MÁS LIMPIA: norma vigente y sus reglamentos ya están en el grafo.
    # -------------------------------------------------------------------------
    {
        "id": "ficha_eas_6480_20",
        "titulo": "Empresas por Acciones Simplificadas - EAS (Ley 6480/2020)",
        "tipo": "ficha_mic",
        "estado": "vigente",
        "fuente": "MIC",
        "nota": ("Montos en guaraníes sujetos a variación (atados a los umbrales MIPYME "
                 "del Decreto 11.453/13). Verificar el monto vigente antes de citarlo."),
        "describe": ["6480", "3998/20", "DGPEJBF 02/22"],
        "texto": """OBJETO: La Ley 6480/2020 crea la Empresa por Acciones Simplificadas (EAS), una nueva PERSONERÍA JURÍDICA diseñada con un enfoque orientado a los emprendedores; un nuevo tipo societario que permite realizar una actividad lucrativa lícita en forma organizada, participando y asumiendo tanto los beneficios como las pérdidas resultantes de esta unidad económica. La EAS es una empresa de capital cuya naturaleza será siempre comercial, con independencia de las actividades previstas en su objeto social. La apertura de una EAS es totalmente EN LÍNEA a través de la web www.eas.mic.gov.py, y se accede únicamente mediante la identidad electrónica del Representante Legal principal de la empresa, de nacionalidad paraguaya o que cuente con cédula paraguaya.

PRINCIPALES BENEFICIOS Y CARACTERÍSTICAS:
- Se tramita totalmente en línea.
- Se constituye en un máximo de 72 horas y con costo CERO, utilizando los estatutos estándar.
- No establece capital mínimo ni máximo para conformarse.
- Establece la separación entre la persona física y la persona jurídica, para que el patrimonio personal del socio (o los socios) permanezca protegido. Los integrantes de la EAS responden hasta el límite de sus aportes comprometidos.
- Permite que las empresas permanezcan y crezcan, generando más empleos.
- Tributa como persona jurídica, de acuerdo con el sector de actividad e ingreso.
- Deben emitir solamente acciones nominales.
- No necesita publicar su creación, convocatorias de asamblea, disolución, etc. en un medio masivo de comunicación, ya que se publica en la página www.eas.mic.gov.py.
- La constitución podrá realizarse por contrato o acto unilateral, por medio de instrumento público o privado con certificación de firmas.
- Adquiere personalidad jurídica (distinta a la de sus integrantes) desde el momento de su inscripción en el Ministerio de Hacienda.
- No se requiere que sea inscrita en el Registro Público de Comercio para poder operar.
- Su inscripción se tramita en el Sistema Unificado de Apertura y Cierre de Empresas (SUACE), mediante un formulario único y un modelo de estatutos sociales.

REQUISITOS:
1. Identidad electrónica del Representante Legal de la empresa.
2. Representante Legal y demás autoridades de la empresa: cédula de identidad paraguaya.
3. Socios: cédula paraguaya, pasaporte, carnet de Radicación Permanente o documento de identidad del país de origen.
4. DOCUMENTOS QUE RESPALDAN EL CAPITAL INTEGRADO, según lo integrado:
   - EFECTIVO: no requiere comprobante si el aporte en efectivo no supera el monto establecido en el Art. 4º del Decreto Nº 11.453/13 (reglamenta la Ley Nº 4.457/2012 para las Micro, Pequeñas y Medianas Empresas). Este monto está sujeto a variación.
   - BOLETA DE DEPÓSITO DE GARANTÍA DEL 20%: únicamente si el aporte en efectivo supera dicho monto. Se deposita en el Banco Nacional de Fomento, cuenta número 948150, a nombre del Ministerio de Industria y Comercio (EAS) Ley Nº 6480/20.
   - BIENES REGISTRABLES (inmuebles, rodados, etc.): escritura pública del bien a nombre del socio que lo integra, consagrándose en el acto constitutivo el valor que se atribuye a los bienes aportados y los antecedentes que justifiquen esa estimación.
   - BIENES NO REGISTRABLES (equipos, mercaderías, etc.): factura comercial del bien a integrar, o inventario de valor firmado por los socios y un contador público nacional.
   - SEMOVIENTES Y OTROS: comprobantes que justifiquen su valoración, a nombre del socio que lo integra.

LAS PERSONAS JURÍDICAS ADEMÁS DEBEN PRESENTAR:
1. Cédula de identidad vigente del presidente o representante legal que tiene uso de la firma (conforme al estatuto).
2. Escritura de constitución de la empresa (socio jurídico).
3. Cédula tributaria.
4. Acta de Directorio (decisión del directorio de constituir una empresa).
5. Datos de la inscripción en el Registro Público.
6. Transcripción de la última asamblea ordinaria.

MARCO LEGAL: Ley Nº 6480/2020 (que crea la Empresa por Acciones Simplificadas EAS); Decreto Nº 3998 del 28 de agosto de 2020; Resolución Nº 623 (reglamenta el proceso de apertura de EAS); Resolución DGPEJyBF Nº 01/2021 (proceso de apertura de una EAS en Abogacía del Tesoro); Resolución DGPEJBF Nº 02/2022 (reglamenta el proceso de apertura de EAS creadas por Ley Nº 6480/2020).""",
    },
]

# -----------------------------------------------------------------------------
# 1) PRE-CHECK: ¿existen en el grafo las normas que las fichas dicen describir?
# -----------------------------------------------------------------------------
numeros = sorted({n for f in FICHAS for n in f["describe"]})
recs, _, _ = driver.execute_query(
    """
    UNWIND $nums AS num
    OPTIONAL MATCH (n) WHERE n.numero = num AND (n:Ley OR n:Decreto OR n:Resolucion)
    RETURN num, n IS NOT NULL AS existe,
           CASE WHEN n IS NULL THEN null ELSE coalesce(n.estado,'sin_dato') END AS estado
    ORDER BY num
    """,
    nums=numeros, database_=NEO4J_DATABASE,
)
print("PRE-CHECK de normas referenciadas por las fichas:")
faltantes = []
for r in recs:
    d = r.data()
    marca = "OK " if d["existe"] else "FALTA"
    print(f"  [{marca}] {d['num']:<15} estado={d['estado']}")
    if not d["existe"]:
        faltantes.append(d["num"])
if faltantes:
    print(f"\n  AVISO: {faltantes} no existen en el grafo -> esas aristas DESCRIBE NO se crearán.")
    print("  (Las fichas igual se cargan; solo pierden ese enlace de vigencia.)")

# -----------------------------------------------------------------------------
# 2) CREAR los nodos Ficha (idempotente) y las aristas DESCRIBE (MATCH-only)
# -----------------------------------------------------------------------------
for f in FICHAS:
    driver.execute_query(
        """
        MERGE (fi:Ficha {id: $id})
        SET fi.titulo = $titulo,
            fi.texto  = $texto,
            fi.tipo   = $tipo,
            fi.estado = $estado,
            fi.fuente = $fuente,
            fi.nota   = $nota
        """,
        id=f["id"], titulo=f["titulo"], texto=f["texto"], tipo=f["tipo"],
        estado=f["estado"], fuente=f["fuente"], nota=f["nota"],
        database_=NEO4J_DATABASE,
    )
    # MATCH-only: si la norma no existe, NO se crea nodo fantasma.
    driver.execute_query(
        """
        MATCH (fi:Ficha {id: $id})
        UNWIND $describe AS num
        MATCH (n) WHERE n.numero = num AND (n:Ley OR n:Decreto OR n:Resolucion)
        MERGE (fi)-[:DESCRIBE]->(n)
        """,
        id=f["id"], describe=f["describe"], database_=NEO4J_DATABASE,
    )
print(f"\n{len(FICHAS)} fichas creadas/actualizadas.")

# -----------------------------------------------------------------------------
# 3) EMBEBER las fichas (mismo modelo y mismo indice que los articulos)
# -----------------------------------------------------------------------------
recs, _, _ = driver.execute_query(
    """
    MATCH (f:Ficha)
    WHERE f.texto IS NOT NULL AND f.embedding IS NULL
    RETURN elementId(f) AS eid, f.titulo AS titulo, f.texto AS texto
    """,
    database_=NEO4J_DATABASE,
)
pendientes = [r.data() for r in recs]
print(f"Fichas pendientes de embeber: {len(pendientes)}")

if pendientes:
    preparados = []
    for row in pendientes:
        # Mismo criterio que los articulos: encabezado de contexto + texto.
        texto_final, _, recortado = recortar_a_limite(f"{row['titulo']}: {row['texto']}")
        if recortado:
            print(f"  AVISO: '{row['titulo']}' se recortó por límite de tokens.")
        preparados.append((row["eid"], texto_final))

    vectores = embeber_lote([t for _, t in preparados])
    filas = [{"eid": eid, "embedding": v} for (eid, _), v in zip(preparados, vectores)]
    escribir_vectores(driver, filas)   # setea el vector Y la etiqueta :Recuperable
    print(f"  {len(filas)} fichas embebidas y guardadas.")

# -----------------------------------------------------------------------------
# 4) VERIFICACIÓN
# -----------------------------------------------------------------------------
recs, _, _ = driver.execute_query(
    """
    MATCH (f:Ficha)
    OPTIONAL MATCH (f)-[:DESCRIBE]->(n)
    RETURN f.titulo AS ficha,
           f.estado AS estado,
           f.embedding IS NOT NULL AS embebida,
           'Recuperable' IN labels(f) AS recuperable,
           collect(n.numero + ' [' + coalesce(n.estado,'sin_dato') + ']') AS describe
    ORDER BY ficha
    """,
    database_=NEO4J_DATABASE,
)
print("\nVERIFICACIÓN:")
for r in recs:
    d = r.data()
    print(f"  {d['ficha']}")
    print(f"    estado={d['estado']} | embebida={d['embebida']} | :Recuperable={d['recuperable']}")
    print(f"    describe -> {', '.join(d['describe'])}")


### Correr el backfill

Primero con `TEST_LIMIT = 20`; verificá y después poné `TEST_LIMIT = None` en la config y volvé a correr la config y esta celda.

In [ ]:
# Parte A — correr el backfill (una sola vez). Usa el 'driver' compartido del Setup.
# TEST_LIMIT=20 (config) hace una prueba; pone None para el corpus completo.
crear_indices(driver)
pendientes = leer_pendientes(driver)
print(f"Articulos pendientes de embeber: {len(pendientes)}")
if pendientes:
    total_tokens = 0
    preparados, recortados = [], []
    for row in pendientes:
        texto_final, n_tok, fue_recortado = recortar_a_limite(texto_con_encabezado(row))
        total_tokens += n_tok
        preparados.append((row["eid"], texto_final))
        if fue_recortado:
            recortados.append(row["eid"])
    costo = total_tokens / 1_000_000 * 0.02
    print(f"Tokens aprox: {total_tokens:,} | costo estimado: ~US${costo:.4f}")
    if recortados:
        print(f"AVISO: {len(recortados)} articulos se recortaron: {recortados[:10]}")
    hechos = 0
    for i in range(0, len(preparados), EMBED_BATCH_SIZE):
        lote = preparados[i:i + EMBED_BATCH_SIZE]
        textos = [t for _, t in lote]
        vectores = embeber_lote(textos)
        filas = [{"eid": eid, "embedding": vec} for (eid, _), vec in zip(lote, vectores)]
        escribir_vectores(driver, filas)
        hechos += len(filas)
        print(f"  {hechos}/{len(preparados)} embebidos y guardados")
        time.sleep(0.2)
verificar(driver)
print("Backfill listo.")

## Parte B — Retriever GraphRAG (uso normal)

### Retriever + pipeline

Vector sobre `recuperable_embedding` → expansión Cypher a la norma, su vigencia y `DEROGA`/`MODIFICA`/`REGLAMENTA`. Construye `retriever` y el pipeline `rag`.

In [5]:
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from neo4j_graphrag.generation import GraphRAG

RETRIEVAL_QUERY = """
OPTIONAL MATCH (norma)-[:TIENE_ARTICULO]->(node)
    WHERE norma:Ley OR norma:Decreto OR norma:Resolucion
WITH node, collect(norma)[0] AS norma
RETURN
    node.texto      AS texto,
    node.numero     AS articulo,
    node.ley_numero AS norma_numero,
    CASE WHEN norma IS NULL THEN null
         ELSE head([l IN labels(norma) WHERE l IN ['Ley','Decreto','Resolucion']]) END AS norma_tipo,
    CASE WHEN norma IS NULL THEN null ELSE norma.nombre_completo END AS norma_titulo,
    CASE WHEN norma IS NULL THEN 'sin_dato' ELSE coalesce(norma.estado,'sin_dato') END AS estado,
    CASE WHEN norma IS NULL THEN null ELSE norma.derogada_por END AS derogada_por,
    CASE WHEN norma IS NULL THEN null ELSE norma.vigente_hasta END AS vigente_hasta,
    CASE WHEN norma IS NULL THEN null ELSE norma.fuente_url END AS fuente_url,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:DEROGA]-(x)     | toString(x.numero) ] END AS derogada_por_rel,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:MODIFICA]-(y)   | toString(y.numero) ] END AS modificada_por_rel,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)<-[:REGLAMENTA]-(d) | toString(d.numero) + ' — ' + coalesce(d.nombre_completo,'') ] END AS reglamentada_por,
    CASE WHEN norma IS NULL THEN [] ELSE [ (norma)-[:REGLAMENTA]->(l) | toString(l.numero) + ' — ' + coalesce(l.nombre_completo,'') ] END AS reglamenta_a
"""

def formatear(record):
    estado = record.get("estado")
    if record.get("derogada_por"):
        marca = f"  [DEROGADA por {record.get('derogada_por')}]"
    elif record.get("vigente_hasta"):
        marca = f"  [vigente hasta {record.get('vigente_hasta')}]"
    elif estado and str(estado).lower() not in ("vigente", "sin_dato"):
        marca = f"  [estado: {estado}]"
    else:
        marca = ""
    tipo = record.get("norma_tipo") or ""
    num  = record.get("norma_numero") or ""
    encabezado = f"{tipo} {num}, Artículo {record.get('articulo')}{marca}".strip()

    extras = []
    if record.get("reglamentada_por"):
        extras.append("Reglamentada por: " + "; ".join(record.get("reglamentada_por")))
    if record.get("reglamenta_a"):
        extras.append("Reglamenta a: " + "; ".join(record.get("reglamenta_a")))
    if record.get("modificada_por_rel"):
        extras.append("Modificada por: " + ", ".join(record.get("modificada_por_rel")))
    extra_txt = ("\n" + " | ".join(extras)) if extras else ""

    return RetrieverResultItem(
        content=f"{encabezado}\n{record.get('texto') or ''}{extra_txt}",
        metadata={
            "norma": num,
            "articulo": record.get("articulo"),
            "estado": estado,
            "derogada_por": record.get("derogada_por"),
            "fuente_url": record.get("fuente_url"),
        },
    )

retriever = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX_NAME,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=formatear,
    neo4j_database=NEO4J_DATABASE,
)
rag = GraphRAG(retriever=retriever, llm=llm, prompt_template=prompt_template)
print("Retriever + pipeline GraphRAG listos.")

Retriever + pipeline GraphRAG listos.


### Probar solo el retriever

Sin LLM (barato). Deberías ver el encabezado, la marca de vigencia si corresponde, y las líneas de reglamenta/modifica cuando existan.

In [6]:
# Probar SOLO el retriever (sin LLM): ver que contexto trae
pregunta = "¿Qué beneficios ofrece el régimen de zonas francas?"
res = retriever.search(query_text=pregunta, top_k=TOP_K)
for i, it in enumerate(res.items, 1):
    print(f"[{i}] {it.metadata}")
    print(it.content)
    print("---")

[1] {'norma': '523/95', 'articulo': 1, 'estado': 'vigente', 'derogada_por': None, 'fuente_url': 'https://www.mef.gov.py/sites/default/files/2025-08/1.-%20Ley%20523-1995_0.pdf'}
Ley 523/95, Artículo 1
Las Zonas Francas son espacios del territorio nacional, localizadas y autorizadas como tales por el Poder Ejecutivo, sujetas al control fiscal, aduanero y administrativo que se establece en la presente ley en las reglamentaciones pertinentes.
---
[2] {'norma': '523/95', 'articulo': 2, 'estado': 'vigente', 'derogada_por': None, 'fuente_url': 'https://www.mef.gov.py/sites/default/files/2025-08/1.-%20Ley%20523-1995_0.pdf'}
Ley 523/95, Artículo 2
Las Zonas Francas deberán instalarse en áreas de propiedad privada, cercadas en forma de garantizar su aislamiento respecto del Territorio Aduanero, con un solo sector de entrada y salida de las mismas. 
Para los efectos de esta ley se entenderá por: 
Territorio aduanero: todo el ámbito terrestre, acuático y aéreo sometido a la soberanía de la Repúbli

### Demo del pipeline (retriever + gpt-4o)

Respuesta anclada al contexto, citando norma/artículo y respetando la vigencia.

In [ ]:
# Demo del pipeline GraphRAG completo (usa 'rag' del bloque del retriever)
pregunta = "¿Qué beneficios ofrece el régimen de zonas francas?"
resp = rag.search(query_text=pregunta, retriever_config={"top_k": TOP_K}, return_context=True)
print(resp.answer)

## Parte C — Baseline vector-only y comparación

El **baseline** es el mismo sistema con el **grafo apagado**: idéntico embedder, LLM, prompt y `top_k`, pero el `retrieval_query` **no expande** al grafo. Cualquier diferencia es atribuible al grafo.

In [7]:
# Baseline vector-only: MISMO todo (embedder, LLM, prompt, top_k), pero el
# retrieval_query NO expande al grafo (solo devuelve el articulo).
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from neo4j_graphrag.generation import GraphRAG

RETRIEVAL_QUERY_BASELINE = """
RETURN node.texto AS texto, node.numero AS articulo, node.ley_numero AS norma_numero
"""

def formatear_baseline(record):
    num = record.get("norma_numero") or ""
    art = record.get("articulo")
    encabezado = f"Norma {num}, Artículo {art}".strip()
    return RetrieverResultItem(
        content=f"{encabezado}\n{record.get('texto') or ''}",
        metadata={"norma": num, "articulo": art},
    )

retriever_baseline = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX_NAME,
    retrieval_query=RETRIEVAL_QUERY_BASELINE,
    embedder=embedder,
    result_formatter=formatear_baseline,
    neo4j_database=NEO4J_DATABASE,
)
rag_baseline = GraphRAG(retriever=retriever_baseline, llm=llm, prompt_template=prompt_template)
print("Baseline + pipeline baseline listos.")

Baseline + pipeline baseline listos.


### Comparar baseline vs GraphRAG

Misma pregunta por los dos. En la consulta testigo de la **Ley 60/90 (derogada)**, el baseline tiende a listarla como vigente (la falla), y el GraphRAG avisa la derogación (el acierto).

In [8]:
# Comparar baseline vs GraphRAG sobre la MISMA pregunta (aisla el aporte del grafo).
# Chequeo de dependencias: si algo falta, te dice qué celda correr (en vez de un NameError).
_faltan = [n for n in ("TOP_K", "rag", "rag_baseline") if n not in globals()]
if _faltan:
    print("⚠ Faltan objetos para correr esta celda:", ", ".join(_faltan))
    if "TOP_K" in _faltan:
        print("   → Corré el SETUP (celda de config): define TOP_K y la conexión.")
    if "rag" in _faltan:
        print("   → Corré la celda del RETRIEVER (Parte B): crea 'retriever' y 'rag' "
              "(termina en 'Retriever + pipeline GraphRAG listos.').")
    if "rag_baseline" in _faltan:
        print("   → Corré la celda del BASELINE (Parte C): crea 'rag_baseline' "
              "(termina en 'Baseline + pipeline baseline listos.').")
    print("   Atajo: Runtime → 'Run before' parado en esta celda corre todo lo de arriba en orden.")
else:
    pregunta = "¿qué incentivos fiscales ofrece la Ley 60/90?"

    print("=== BASELINE (vector-only, grafo apagado) ===")
    print(rag_baseline.search(query_text=pregunta, retriever_config={"top_k": TOP_K}).answer)

    print("\n=== GraphRAG (con grafo) ===")
    print(rag.search(query_text=pregunta, retriever_config={"top_k": TOP_K}).answer)

=== BASELINE (vector-only, grafo apagado) ===


La Ley N° 60/90 tiene como objetivo promover e incrementar las inversiones de capital de origen nacional y/o extranjero, otorgando beneficios de carácter fiscal a las personas físicas y jurídicas radicadas en Paraguay. Los incentivos fiscales están destinados a proyectos que busquen:

a) El acrecentamiento de la producción de bienes y servicios.
b) La creación de fuentes de trabajo permanente.
c) El fomento de las exportaciones y la sustitución de importaciones.
d) La incorporación de tecnologías que permitan aumentar la eficiencia productiva y posibiliten la mayor y mejor utilización de materias primas, mano de obra y recursos energéticos nacionales.
e) La inversión y reinversión de utilidades en bienes de capital.

Estos beneficios son irrevocables, salvo en los casos previstos en el Artículo 16, incisos a), b), c) y d) de la normativa correspondiente (Norma 60/90, Artículo 1 y Artículo 25).

=== GraphRAG (con grafo) ===


La Ley 60/90 ha sido derogada por la Ley 7548/25, por lo que no está vigente. Por lo tanto, no puedo presentar los incentivos fiscales de la Ley 60/90 como vigentes. Si necesitas información sobre incentivos fiscales actuales, te recomiendo revisar las disposiciones de la Ley 7548/25 y sus artículos correspondientes.


## Parte D — Agente conversacional (LangGraph · Etapas 1-2-3)

**Requisitos:** haber corrido el **Setup** y la celda del **retriever (Parte B)** — usa `retriever`, `llm` y `TOP_K`.

Grafo: **reescribir** (Etapa 2: consulta autónoma con el historial) → **recuperar** → **graduar** (Etapa 3: el LLM juzga si el contexto alcanza) → según el veredicto: **responder** (genera con memoria, Etapa 1), **reformular** → volver a **recuperar** (un reintento), o **sin_contexto** (respuesta honesta si no hay info). Checkpointer con `thread_id` para la memoria. Solo `langgraph`.

La generación (`responder`) usa `retriever` + `llm` por separado (no `rag.search`) para poder graduar el contexto antes de responder; la vigencia sigue expuesta porque el contexto trae las marcas `[DEROGADA…]` del retriever.

### Grafo (reescribir → recuperar → graduar → responder / reformular / sin_contexto)

In [9]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM = (
    "Sos un asistente legal sobre derecho de inversiones de Paraguay. "
    "Respondé usando SOLO el contexto provisto. Citá la norma y el artículo en cada afirmación. "
    "Si una norma figura como DEROGADA o con estado distinto de vigente, aclaralo y NO la "
    "presentes como vigente. Si el contexto no alcanza, decilo."
)

class EstadoConv(TypedDict):
    messages: Annotated[list, operator.add]
    consulta: str      # consulta (reescrita) usada para buscar
    context: str       # contexto recuperado
    intentos: int      # cuantas veces se busco (corta el bucle)
    relevante: bool    # veredicto del grading

# --- Etapa 2: reescritura history-aware ---
def reescribir(state: EstadoConv):
    if len(state["messages"]) <= 1:
        return {"consulta": state["messages"][-1]["content"], "intentos": 0}
    historial = "\n".join(f'{m["role"]}: {m["content"]}' for m in state["messages"][:-1])
    pregunta = state["messages"][-1]["content"]
    prompt = (
        "Dada la conversación previa y una pregunta de seguimiento, reescribí la pregunta como "
        "una consulta AUTÓNOMA que se entienda sin el historial (resolvé 'esa ley', 'ese régimen', "
        "'ahí'). Si ya es autónoma, devolvela igual. Devolvé SOLO la consulta, sin comillas.\n\n"
        f"# Conversación:\n{historial}\n\n# Pregunta:\n{pregunta}\n\n# Consulta autónoma:"
    )
    return {"consulta": llm.invoke(prompt).content.strip(), "intentos": 0}

def recuperar(state: EstadoConv):
    res = retriever.search(query_text=state["consulta"], top_k=TOP_K)
    context = "\n\n".join(it.content for it in res.items)
    return {"context": context, "intentos": state.get("intentos", 0) + 1}

# --- Etapa 3: grading + re-busqueda (corrective RAG) ---
def graduar(state: EstadoConv):
    # OJO: sin la aclaracion de abajo, el grader interpreta "norma DEROGADA" como
    # "contexto insuficiente" y manda al fallback, tapando la advertencia de vigencia
    # (que es justo el aporte del sistema). Bug real detectado con la Ley 5102.
    prompt = (
        "Decidí si el CONTEXTO alcanza para responder la PREGUNTA de forma fundamentada. "
        "Respondé SOLO con 'SI' o 'NO'.\n\n"
        "IMPORTANTE: si el contexto contiene artículos de la norma consultada, alcanza — "
        "AUNQUE la norma figure como DEROGADA, modificada o no vigente. Advertir que una norma "
        "está derogada (y por cuál fue reemplazada) es una respuesta VÁLIDA y útil, no una falta "
        "de información. Respondé 'NO' solo si el contexto es de otra materia o no tiene nada "
        "que ver con la pregunta.\n\n"
        f"# Pregunta:\n{state['consulta']}\n\n# Contexto:\n{state['context'][:4000]}\n\n# ¿Alcanza? (SI/NO):"
    )
    veredicto = llm.invoke(prompt).content.strip().upper()
    return {"relevante": veredicto.startswith("SI")}

def decidir(state: EstadoConv):
    # Router: SOLO lee el estado y devuelve el nombre del proximo nodo (sin llamar al LLM aca)
    if state["relevante"]:
        return "responder"
    if state["intentos"] < 2:     # un reintento con reformulacion
        return "reformular"
    return "sin_contexto"

def reformular(state: EstadoConv):
    prompt = (
        "La búsqueda anterior no trajo contexto suficiente. Reformulá la consulta con otros "
        "términos o de forma más general para mejorar la recuperación. Devolvé SOLO la nueva consulta.\n\n"
        f"# Consulta anterior:\n{state['consulta']}"
    )
    return {"consulta": llm.invoke(prompt).content.strip()}

def responder(state: EstadoConv):
    history = state["messages"][:-1]
    resp = llm.invoke(
        input=f"# Contexto:\n{state['context']}\n\n# Pregunta:\n{state['consulta']}",
        message_history=history,
        system_instruction=SYSTEM,
    )
    return {"messages": [{"role": "assistant", "content": resp.content}]}

def sin_contexto(state: EstadoConv):
    msg = ("No encontré en la base normativa cargada información suficiente para responder eso "
           "con fundamento. ¿Podés reformular la pregunta o dar más detalle?")
    return {"messages": [{"role": "assistant", "content": msg}]}

builder = StateGraph(EstadoConv)
builder.add_node("reescribir", reescribir)
builder.add_node("recuperar", recuperar)
builder.add_node("graduar", graduar)
builder.add_node("reformular", reformular)
builder.add_node("responder", responder)
builder.add_node("sin_contexto", sin_contexto)

builder.add_edge(START, "reescribir")
builder.add_edge("reescribir", "recuperar")
builder.add_edge("recuperar", "graduar")
builder.add_conditional_edges("graduar", decidir, {
    "responder": "responder",
    "reformular": "reformular",
    "sin_contexto": "sin_contexto",
})
builder.add_edge("reformular", "recuperar")   # bucle: vuelve a buscar con la consulta reformulada
builder.add_edge("responder", END)
builder.add_edge("sin_contexto", END)

chat_graph = builder.compile(checkpointer=InMemorySaver())
print("Grafo conversacional (memoria + reescritura + corrective RAG) listo.")

Grafo conversacional (memoria + reescritura + corrective RAG) listo.


### Demo

Se imprime la **consulta usada** y cuántas **búsquedas** hizo. Muestra los tres caminos: conversación normal (memoria + reescritura), y una pregunta fuera de dominio que dispara el grading → reformulación → respuesta honesta de que no hay info suficiente.

In [10]:
import uuid

def preguntar(chat_id, texto):
    out = chat_graph.invoke({"messages": [{"role": "user", "content": texto}]},
                            {"configurable": {"thread_id": chat_id}})
    print("Usuario:", texto)
    print("  (consulta usada:", out["consulta"], "| busquedas:", out["intentos"], ")")
    print("Bot:", out["messages"][-1]["content"], "\n")

# 1) Conversación normal: memoria (Etapa 1) + reescritura (Etapa 2)
chat = str(uuid.uuid4())
preguntar(chat, "¿Qué es el régimen de zonas francas?")
preguntar(chat, "¿Qué actividades permite?")

# 2) Norma DEROGADA: el sistema debe ADVERTIR la derogación (no decir "no sé")
preguntar(str(uuid.uuid4()), "¿qué establece la Ley 5102 sobre alianza público-privada?")

# 3) Fuera de dominio: grading NO -> reformula -> fallback honesto (Etapa 3)
preguntar(str(uuid.uuid4()), "¿Cuál es la mejor receta de sopa paraguaya?")

Usuario: ¿Qué es el régimen de zonas francas?
  (consulta usada: ¿Qué es el régimen de zonas francas? | busquedas: 1 )
Bot: El régimen de zonas francas en Paraguay, según la Ley 523/95, se refiere a espacios del territorio nacional que son localizados y autorizados por el Poder Ejecutivo. Estas zonas están sujetas a un control fiscal, aduanero y administrativo específico establecido por la ley y sus reglamentaciones pertinentes (Ley 523/95, Artículo 1). Las zonas francas deben instalarse en áreas de propiedad privada y estar cercadas para garantizar su aislamiento del Territorio Aduanero, teniendo un solo sector de entrada y salida (Ley 523/95, Artículo 2).

Dentro de las zonas francas, se pueden desarrollar actividades comerciales, industriales y de servicios. Las actividades comerciales incluyen la internación de bienes para su intermediación sin transformación, mientras que las industriales se centran en la fabricación de bienes para exportación mediante la transformación de materia

Usuario: ¿Qué actividades permite?
  (consulta usada: ¿Qué actividades permite el régimen de zonas francas en Paraguay según la Ley 523/95? | busquedas: 1 )
Bot: Según la Ley 523/95, el régimen de zonas francas en Paraguay permite desarrollar las siguientes actividades:

a) **Comerciales**: Estas actividades implican la internación de bienes para su intermediación sin que sufran transformación o modificación. Incluyen el depósito, selección, clasificación, manipulación, y mezcla de mercaderías o materias primas (Ley 523/95, Artículo 3, inciso a).

b) **Industriales**: Se refiere a la fabricación de bienes destinados a la exportación mediante la transformación de materias primas y/o productos semielaborados de origen nacional o importado. También incluye actividades clasificadas como ensamblaje (Ley 523/95, Artículo 3, inciso b).

c) **Servicios**: Comprende reparaciones y mantenimiento de equipos y maquinarias. Además, otros servicios destinados al mercado internacional pueden ser auto

Usuario: ¿qué establece la Ley 5102 sobre alianza público-privada?
  (consulta usada: ¿qué establece la Ley 5102 sobre alianza público-privada? | busquedas: 1 )
Bot: La Ley 5102, que ha sido derogada por la Ley 7452/25, establecía normas y mecanismos para promover las inversiones en infraestructura pública y la prestación de servicios a través de la participación público-privada. Esta ley contemplaba la figura jurídica de los contratos de participación público-privada, la iniciativa privada y regulaba el uso de fideicomisos para estos fines (Artículo 1). Además, los contratos de participación público-privada debían regirse por los términos del contrato, las disposiciones de la ley y la reglamentación del Poder Ejecutivo (Artículo 5). También se especificaba que estos contratos debían establecer los riesgos, compromisos y beneficios asumidos por el Estado y el participante privado (Artículo 4). Sin embargo, es importante destacar que esta ley ya no está vigente. 



Usuario: ¿Cuál es la mejor receta de sopa paraguaya?
  (consulta usada: Receta tradicional de sopa paraguaya | busquedas: 2 )
Bot: No encontré en la base normativa cargada información suficiente para responder eso con fundamento. ¿Podés reformular la pregunta o dar más detalle? 



## Notas

- `TOP_K` = cuántos artículos recupera (no la ventana de contexto); más alto = más contexto y costo.
- Warning `db.index.vector.queryNodes` deprecado: inofensivo. Descomentá la línea de `logging` en la config para silenciarlo.
- Etapas siguientes de LangGraph: reescritura de consulta (Etapa 2) y grading + re-búsqueda (Etapa 3).
- Al terminar, `driver.close()` si no vas a seguir usando la conexión.